# AeroRUL -- Phase 2: RUL Formulation, Target Construction & Leakage-Safe Splitting

**Objective**: Construct scientifically correct RUL targets, investigate clipping, establish leakage-safe engine-level train/validation splitting, and investigate condition-aware normalization.

---

## 1. Phase 2 Objective

Phase 1 established that the C-MAPSS dataset is clean and well-structured. Phase 2 prepares it for ML:

1. **RUL Target**: `RUL = failure_cycle - current_cycle`
2. **RUL Clipping**: Investigate capping RUL at various thresholds
3. **Engine-Level Split**: No engine in both train and validation
4. **Operating-Condition Normalization**: For multi-condition subsets (FD002/FD004)

## 2. Load Phase 1 Data

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import SENSOR_COLUMNS, OP_SETTING_COLUMNS, RUL_CLIP_VALUES
from src.data_loader import load_cmapss, load_all_subsets
from src.rul import generate_rul_targets, validate_rul_targets, rul_statistics, clip_rul, rul_clipping_comparison
from src.splitting import split_by_engine, validate_split
from src.normalization import identify_operating_regimes, fit_regime_normalization, apply_regime_normalization, regime_summary, regime_sensor_analysis

pd.set_option('display.max_columns', 30)
%matplotlib inline

# Load all subsets
all_data = load_all_subsets()
for subset, data in all_data.items():
    print(f"{subset}: train={data['train'].shape}, test={data['test'].shape}")

## 3. Generate RUL

In [ ]:
# Generate RUL targets for all subsets
all_with_rul = {}
for subset, data in sorted(all_data.items()):
    df_rul = generate_rul_targets(data['train'])
    all_with_rul[subset] = df_rul
    print(f"{subset}: {df_rul['unit_id'].nunique()} engines, RUL range [{df_rul['rul'].min()}, {df_rul['rul'].max()}]")

# Show example
print("\nFD001 Engine 1 (first/last rows):")
e1 = all_with_rul['FD001'][all_with_rul['FD001']['unit_id'] == 1][['unit_id', 'cycle', 'rul']]
print(e1.head())
print('...')
print(e1.tail())

## 4. Validate RUL

In [ ]:
# Validate RUL targets for all subsets
for subset, df_rul in sorted(all_with_rul.items()):
    v = validate_rul_targets(df_rul)
    print(f"\n{subset} RUL Validation:")
    print(f"  All valid:               {v['all_valid']}")
    print(f"  RUL >= 0:                {v['rul_non_negative']}")
    print(f"  min(RUL)==0 per engine:  {v['min_rul_zero_per_engine']}")
    print(f"  max(RUL) correct:        {v['max_rul_correct']}")
    print(f"  Decreases by 1:          {v['rul_decreases_by_one']}")
    print(f"  RUL+cycle invariant:     {v['rul_plus_cycle_invariant']}")

## 5. RUL Distribution

In [ ]:
# RUL statistics
for subset, df_rul in sorted(all_with_rul.items()):
    stats = rul_statistics(df_rul)
    print(f"{subset}: min={stats['min_rul']}, max={stats['max_rul']}, mean={stats['mean_rul']}, median={stats['median_rul']}")

## 6. RUL Clipping Experiments

In [ ]:
# Clipping comparison for FD001
comp = rul_clipping_comparison(all_with_rul['FD001'])
print("FD001 -- RUL Clipping Comparison:")
comp

In [ ]:
# Visual: Raw vs Clipped RUL
df = all_with_rul['FD001']
engine = df[df['unit_id'] == 1].sort_values('cycle')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(engine['cycle'], engine['rul'], label='Raw RUL', linewidth=2)
for cap in [125, 100, 75]:
    ax.plot(engine['cycle'], engine['rul'].clip(upper=cap), label=f'Clipped ({cap})', linestyle='--', alpha=0.8)
ax.set_xlabel('Cycle')
ax.set_ylabel('RUL')
ax.set_title('FD001 Engine 1 -- Raw vs Clipped RUL', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Engine-Level Train/Validation Split

In [ ]:
# Split all subsets
all_splits = {}
for subset, df_rul in sorted(all_with_rul.items()):
    train_df, val_df = split_by_engine(df_rul)
    all_splits[subset] = (train_df, val_df)
    sv = validate_split(train_df, val_df, df_rul)
    print(f"\n{subset} Split:")
    print(f"  Train: {sv['n_train_engines']} engines, {sv['n_train_rows']} rows")
    print(f"  Val:   {sv['n_val_engines']} engines, {sv['n_val_rows']} rows")
    print(f"  No overlap: {sv['no_engine_overlap']}")
    print(f"  Rows preserved: {sv['rows_preserved']}")

## 8. Split Validation

In [ ]:
# Verify leakage-free split
for subset, (train_df, val_df) in sorted(all_splits.items()):
    train_engines = set(train_df['unit_id'].unique())
    val_engines = set(val_df['unit_id'].unique())
    overlap = train_engines & val_engines
    print(f"{subset}: intersection = {overlap if overlap else 'EMPTY SET'} -> {'PASS' if not overlap else 'FAIL'}")

## 9. Operating-Regime Analysis

In [ ]:
# Regime analysis for multi-condition subsets
for subset in ['FD001', 'FD002', 'FD003', 'FD004']:
    train_df = all_splits[subset][0]
    regimes = identify_operating_regimes(train_df)
    summary = regime_summary(train_df, regimes)
    print(f"\n{subset}: {len(summary)} operating regimes")
    if len(summary) <= 10:
        print(summary.to_string(index=False))

## 10. Condition-Aware Normalization

In [ ]:
# Normalization for FD002 (multi-condition)
train_fd002, val_fd002 = all_splits['FD002']
regimes_fd002 = identify_operating_regimes(train_fd002)
norm_params = fit_regime_normalization(train_fd002, regimes_fd002)

train_fd002_norm = apply_regime_normalization(train_fd002, norm_params, regimes_fd002)

# Verify validation uses training params
val_regimes = identify_operating_regimes(val_fd002)
val_fd002_norm = apply_regime_normalization(val_fd002, norm_params, val_regimes)

print(f"FD002 Training: {len(train_fd002_norm)} rows normalized")
print(f"FD002 Validation: {len(val_fd002_norm)} rows normalized (using training params)")
print(f"\nNormalized sensor_2 stats (training, per regime):")
for regime in sorted(regimes_fd002.unique())[:3]:
    mask = regimes_fd002 == regime
    m = train_fd002_norm.loc[mask, 'sensor_2_norm'].mean()
    s = train_fd002_norm.loc[mask, 'sensor_2_norm'].std()
    print(f"  {regime}: mean={m:.4f}, std={s:.4f}")

## 11. Raw vs Normalized Sensor Analysis

In [ ]:
# Compare raw vs normalized for FD002
engines = sorted(train_fd002['unit_id'].unique())[:3]
sensor = 'sensor_2'

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for eid in engines:
    mask = train_fd002['unit_id'] == eid
    raw = train_fd002.loc[mask].sort_values('cycle')
    norm = train_fd002_norm.loc[mask].sort_values('cycle')
    axes[0].plot(raw['cycle'], raw[sensor], linewidth=0.8, alpha=0.8, label=f'Engine {eid}')
    axes[1].plot(norm['cycle'], norm[f'{sensor}_norm'], linewidth=0.8, alpha=0.8, label=f'Engine {eid}')

axes[0].set_title(f'FD002 {sensor} (Raw)', fontweight='bold')
axes[0].set_xlabel('Cycle'); axes[0].set_ylabel('Value'); axes[0].legend(fontsize=8)
axes[1].set_title(f'FD002 {sensor} (Condition-Normalized)', fontweight='bold')
axes[1].set_xlabel('Cycle'); axes[1].set_ylabel('Normalized Value'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("\nKey observation: After normalization, the operating-condition jumps are removed,")
print("making the underlying degradation trend more visible.")

## 12. Phase 2 Findings

### Summary

1. **RUL targets are correct**: All 5 validation checks pass for all subsets.
2. **RUL clipping**: Higher caps preserve more information but increase target variance. The optimal threshold should be determined in Phase 3 using model performance.
3. **Engine-level split**: 80/20 train/validation with zero engine overlap confirmed.
4. **Operating regimes**: FD001/FD003 have ~1 regime, FD002/FD004 have 6 regimes.
5. **Condition-aware normalization**: Makes degradation trends clearer in multi-condition subsets by removing operating-condition effects.

### Leakage Prevention
- No engine appears in both train and validation
- Normalization parameters fitted on training data only
- Validation/test transformed using training statistics

### Next Phase
Phase 3: Sliding-window construction and temporal feature engineering.